# Team Ensemble — Blending Individual Submissions

Combine each team member's best individual submission into a single prediction.

## Why ensembling works

Each member trained a different model (different features, different algorithm, different HPO).  
Even if one model is weaker individually, if its **errors are different** from others,  
blending reduces overall error — the same reason stacking XGB+LGBM+ET beats any single model.

## Three blend strategies (all tried below)

| Strategy | How | Best when |
|---|---|---|
| Equal weight | Simple average | Models are similarly strong |
| Weighted by RMSE | Better score = more weight (1/RMSE²) | Models differ significantly in strength |
| Rank average | Average prediction ranks | Models predict at different scales |

**Rule of thumb:** Submit the equal-weight blend first. If it beats everyone's individual best, you're done.

---
## 1. Load Team Submissions

Update `TEAM_FILES` with each member's **best** submission and their Kaggle RMSE.

In [15]:
import pandas as pd
import numpy as np

SUBMISSION_DIR = '../outputs/submissions/Team Submission/'

# Format: (filepath, kaggle_rmse, member_name)
# 2026-05-01 round — 6 members
# Note: Lai file is missing 2 row IDs; filled with Lai's mean below
TEAM_FILES = [
    (SUBMISSION_DIR + '20260430_Lanson.csv',         21591, 'Lanson'),
    (SUBMISSION_DIR + '20260430_21831_likhong.csv',  21831, 'Likhong'),
    (SUBMISSION_DIR + '20260501_22536_Lai.csv',      22536, 'Lai'),
    (SUBMISSION_DIR + '20260501_21343_ben.csv',      21343, 'Ben'),
    (SUBMISSION_DIR + '20260430_23000_hueyling.csv', 23000, 'Hueyling'),
    (SUBMISSION_DIR + '20260429_21827_MengHai.csv',  21827, 'MengHai'),
]

# Load all submissions
dfs = {}
for fpath, rmse_score, name in TEAM_FILES:
    df = pd.read_csv(fpath)
    dfs[name] = df.set_index(df.columns[0])[df.columns[1]]
    print(f'{name:12s} | RMSE: ${rmse_score:>7,.0f} | Rows: {len(df)} | Price range: ${df.iloc[:,1].min():,.0f}-${df.iloc[:,1].max():,.0f}')

# Align all on same index
sample = pd.read_csv('../outputs/submissions/sample_sub_reg.csv')
pred_df = pd.DataFrame({name: s for name, s in dfs.items()}).reindex(sample['Id'])

# Fix Lai's 2 missing rows
pred_df['Lai'] = pred_df['Lai'].fillna(pred_df['Lai'].mean())

print(f'\nAlignment check — any nulls: {pred_df.isna().sum().sum()}')
print(pred_df.head(3))

Lanson       | RMSE: $ 21,591 | Rows: 16735 | Price range: $172,430-$1,165,476
Likhong      | RMSE: $ 21,831 | Rows: 16735 | Price range: $175,395-$1,173,454
Lai          | RMSE: $ 22,536 | Rows: 16735 | Price range: $177,114-$1,156,460
Ben          | RMSE: $ 21,343 | Rows: 16735 | Price range: $181,952-$1,127,777
Hueyling     | RMSE: $ 23,000 | Rows: 16735 | Price range: $176,229-$1,168,153
MengHai      | RMSE: $ 21,827 | Rows: 16735 | Price range: $179,967-$1,159,933

Alignment check — any nulls: 0
          Lanson        Likhong            Lai       Ben       Hueyling  \
Id                                                                        
114982  375589.0  371999.028972  374727.788170  362037.0  377469.917045   
95653   446548.0  461949.633152  457542.269609  450734.0  446163.681313   
40303   352065.0  318768.922219  353400.734878  348856.0  348623.889176   

              MengHai  
Id                     
114982  374913.849779  
95653   447025.613222  
40303   354681.683461 

---
## 2. Correlation Between Predictions

Lower correlation = more potential gain from blending.

In [16]:
print('Prediction correlation matrix:')
print(pred_df.corr().round(4))
print('\nNote: correlation < 0.99 = meaningful diversity for blending')

Prediction correlation matrix:
          Lanson  Likhong     Lai     Ben  Hueyling  MengHai
Lanson    1.0000   0.9954  0.9984  0.9921    0.9975   0.9989
Likhong   0.9954   1.0000  0.9953  0.9911    0.9947   0.9958
Lai       0.9984   0.9953  1.0000  0.9927    0.9974   0.9983
Ben       0.9921   0.9911  0.9927  1.0000    0.9918   0.9929
Hueyling  0.9975   0.9947  0.9974  0.9918    1.0000   0.9978
MengHai   0.9989   0.9958  0.9983  0.9929    0.9978   1.0000

Note: correlation < 0.99 = meaningful diversity for blending


---
## 3. Strategy 1 — Equal Weight Blend

In [17]:
blend_equal = pred_df.mean(axis=1)
print(f'Equal-weight blend:')
print(f'  Mean price:  ${blend_equal.mean():,.0f}')
print(f'  Price range: ${blend_equal.min():,.0f} – ${blend_equal.max():,.0f}')

Equal-weight blend:
  Mean price:  $448,883
  Price range: $177,541 – $1,156,244


---
## 4. Strategy 2 — Weighted by Kaggle RMSE (1/RMSE²)

In [18]:
rmse_scores = {name: rmse for _, rmse, name in TEAM_FILES}
weights_raw = {name: 1 / (rmse**2) for name, rmse in rmse_scores.items()}
total_w     = sum(weights_raw.values())
weights     = {name: w / total_w for name, w in weights_raw.items()}

print('Normalised weights (1/RMSE²):')
for name, w in weights.items():
    print(f'  {name:12s}: {w:.4f}  (RMSE ${rmse_scores[name]:,.0f})')

blend_weighted = sum(pred_df[name] * w for name, w in weights.items())
print(f'\nWeighted blend:')
print(f'  Mean price:  ${blend_weighted.mean():,.0f}')
print(f'  Price range: ${blend_weighted.min():,.0f} – ${blend_weighted.max():,.0f}')

Normalised weights (1/RMSE²):
  Lanson      : 0.1730  (RMSE $21,591)
  Likhong     : 0.1693  (RMSE $21,831)
  Lai         : 0.1588  (RMSE $22,536)
  Ben         : 0.1771  (RMSE $21,343)
  Hueyling    : 0.1525  (RMSE $23,000)
  MengHai     : 0.1693  (RMSE $21,827)

Weighted blend:
  Mean price:  $448,898
  Price range: $177,583 – $1,155,746


---
## 5. Strategy 3 — Rank Averaging

Average the rank (position) of each prediction rather than the raw value.  
More robust when members' models predict at very different price scales.

In [19]:
# Convert each member's predictions to ranks, then average ranks, then map back to prices
ranks_df = pred_df.rank()
avg_rank  = ranks_df.mean(axis=1).rank()  # final rank from averaged ranks

# Map averaged rank back to price scale using equal-weight blend as reference
ref_sorted  = blend_equal.sort_values().values
rank_sorted = avg_rank.sort_values().index
blend_rank  = pd.Series(ref_sorted, index=rank_sorted).reindex(sample['Id'])

print(f'Rank-average blend:')
print(f'  Mean price:  ${blend_rank.mean():,.0f}')
print(f'  Price range: ${blend_rank.min():,.0f} – ${blend_rank.max():,.0f}')

Rank-average blend:
  Mean price:  $448,883
  Price range: $177,541 – $1,156,244


---
## 6. Compare All Strategies

In [20]:
individual_best = min(rmse_scores.values())
best_member     = min(rmse_scores, key=rmse_scores.get)

print('=== Blend Strategy Comparison ===')
print(f'Individual best: ${individual_best:,.0f} ({best_member})')
print()
print('Individual predictions:')
for name, rmse in sorted(rmse_scores.items(), key=lambda x: x[1]):
    print(f'  {name:12s}: ${rmse:>7,.0f} (Kaggle)')
print()
print('Blends (submit to Kaggle to know true RMSE):')
print(f'  Equal weight : mean=${blend_equal.mean():,.0f}')
print(f'  RMSE weighted: mean=${blend_weighted.mean():,.0f}')
print(f'  Rank average : mean=${blend_rank.mean():,.0f}')
print()
print('Recommendation: start with equal-weight blend.')

=== Blend Strategy Comparison ===
Individual best: $21,343 (Ben)

Individual predictions:
  Ben         : $ 21,343 (Kaggle)
  Lanson      : $ 21,591 (Kaggle)
  MengHai     : $ 21,827 (Kaggle)
  Likhong     : $ 21,831 (Kaggle)
  Lai         : $ 22,536 (Kaggle)
  Hueyling    : $ 23,000 (Kaggle)

Blends (submit to Kaggle to know true RMSE):
  Equal weight : mean=$448,883
  RMSE weighted: mean=$448,898
  Rank average : mean=$448,883

Recommendation: start with equal-weight blend.


---
## 7. Save Ensemble Submissions

In [21]:
date = '20260501'

def save_sub(predictions, filename, directory):
    sub = pd.DataFrame({'Id': sample['Id'], 'Predicted': predictions.reindex(sample['Id']).values})
    path = directory + filename
    sub.to_csv(path, index=False)
    print(f'Saved: {filename}  |  mean=${predictions.mean():,.0f}  |  rows={len(sub)}')

print('=== Team Submission folder ===')
save_sub(blend_equal,    f'{date}_6member_equal.csv',    SUBMISSION_DIR)
save_sub(blend_weighted, f'{date}_6member_wt.csv',       SUBMISSION_DIR)
save_sub(blend_rank,     f'{date}_6member_rank.csv',     SUBMISSION_DIR)

print()
print('--- Submission priority ---')
print(f'1. {date}_6member_wt.csv     — weighted (1/RMSE^2); Ben highest, Lanson 2nd')
print(f'2. {date}_6member_equal.csv  — equal weight; maximum diversity')
print(f'3. {date}_6member_rank.csv   — rank avg; most robust to scale differences')

=== Team Submission folder ===
Saved: 20260501_6member_equal.csv  |  mean=$448,883  |  rows=16735
Saved: 20260501_6member_wt.csv  |  mean=$448,898  |  rows=16735
Saved: 20260501_6member_rank.csv  |  mean=$448,883  |  rows=16735

--- Submission priority ---
1. 20260501_6member_wt.csv     — weighted (1/RMSE^2); Ben highest, Lanson 2nd
2. 20260501_6member_equal.csv  — equal weight; maximum diversity
3. 20260501_6member_rank.csv   — rank avg; most robust to scale differences


---
## 8. Submitting Strategy

1. **Submit `sub_ensemble_equal.csv` first** — simplest, usually best
2. If it beats the individual best → great, you're done
3. If not, try `sub_ensemble_weighted.csv` (de-emphasises weaker members)
4. `sub_ensemble_rank.csv` is a tiebreaker if the others are similar

### As more members improve their models

Update `TEAM_FILES` in Cell 1 with each member's latest best file and re-run from Section 1.
The ensemble typically keeps improving as individual models improve.

### Key insight

If two members have very different approaches (e.g. one uses deep feature engineering,  
one uses raw features with aggressive HPO), their errors will be less correlated  
and the blend will improve more than if everyone used the same approach.